# S11-18 — Feature Importance CWRU — Scénario by_fault_type

| Champ | Valeur |
|-------|--------|
| **Expériences** | exp_100 (KMeans), exp_103 (Mahalanobis), exp_106 (EWC), exp_109 (HDC) |
| **Scénario** | by_fault_type : Ball → Inner Race → Outer Race |
| **Features** | 9 features temporelles (max, min, mean, sd, rms, skewness, kurtosis, crest, form) |
| **Sprint** | Sprint 11 — S11-18 |


## Section 1 — Setup, imports et chargement des résultats

In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path('.').resolve()
if _cwd.name == 'cwru_feature_importance':
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == 'cl_eval':
    os.chdir(_cwd.parent.parent)

sys.path.insert(0, str(Path('.').resolve()))

from src.evaluation.feature_importance import (
    FEATURE_NAMES_CWRU,
    plot_feature_importance,
    plot_feature_importance_comparison,
)

FIGURES_DIR = Path("notebooks/figures/cl_evaluation/cwru_feature_importance")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPS = {
    "KMeans":      "experiments/exp_100/results/feature_importance.json",
    "Mahalanobis": "experiments/exp_103/results/feature_importance.json",
    "EWC":         "experiments/exp_106/results/feature_importance.json",
    "HDC":         "experiments/exp_109/results/feature_importance.json",
}

data = {model: json.load(open(path)) for model, path in EXPS.items()}
TASKS = ["ball", "inner_race", "outer_race"]
print("Chargement OK — modèles :", list(data.keys()))
print("Features :", FEATURE_NAMES_CWRU)


Chargement OK — modèles : ['KMeans', 'Mahalanobis', 'EWC', 'HDC']
Features : ['max', 'min', 'mean', 'sd', 'rms', 'skewness', 'kurtosis', 'crest', 'form']


## Section 2 — Tableau récapitulatif global (ranking par modèle)

In [2]:
rows = {}
for model, d in data.items():
    scores = d["global"]["permutation_importance"]
    rows[model] = {f: scores.get(f, 0.0) for f in FEATURE_NAMES_CWRU}

df = pd.DataFrame(rows).T  # modèles en lignes, features en colonnes
df_rank = df.rank(axis=1, ascending=False).astype(int)

print("=== Scores d'importance (permutation globale) ===")
display(df.round(4))
print()
print("=== Rang par feature (1 = plus importante) ===")
display(df_rank)


=== Scores d'importance (permutation globale) ===


,max,min,mean,sd,rms,skewness,kurtosis,crest,form
KMeans,-0.1710,-0.1745,-0.0065,-0.1870,-0.1827,0.0026,-0.0325,-0.0035,-0.1372
Mahalanobis,-0.4944,-0.4835,-0.2108,-0.7368,-0.7286,-0.0528,-0.2268,-0.1818,-0.2736
EWC,0.0277,0.0277,-0.0022,0.0221,0.0481,0.0009,0.0576,-0.0022,0.0286
HDC,0.0108,0.0061,0.0022,-0.0078,-0.0113,-0.0039,0.0069,-0.0035,0.0238



=== Rang par feature (1 = plus importante) ===


,max,min,mean,sd,rms,skewness,kurtosis,crest,form
KMeans,6,7,3,9,8,1,4,2,5
Mahalanobis,7,6,3,9,8,1,4,2,5
EWC,4,4,8,6,2,7,1,9,3
HDC,2,4,5,8,9,7,3,6,1


## Section 3 — Barplot groupé global (4 modèles × 9 features)

In [3]:
importances_dict = {
    model: d["global"]["permutation_importance"]
    for model, d in data.items()
}

fig = plot_feature_importance_comparison(
    importances_dict,
    FEATURE_NAMES_CWRU,
    title="Importance globale — CWRU by_fault_type (4 modèles)",
)
fig.savefig(FIGURES_DIR / "by_fault_type_global_comparison.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_fault_type_global_comparison.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/cwru_feature_importance/by_fault_type_global_comparison.png


/tmp/ipykernel_95591/1640101510.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4 — Heatmap per-task (3 tâches × 9 features) par modèle

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle("Importance par tâche — CWRU by_fault_type", fontsize=13, fontweight="bold")

for ax, (model, d) in zip(axes, data.items()):
    mat = np.array([
        [d["per_task"][t]["permutation_importance"].get(f, 0.0) for f in FEATURE_NAMES_CWRU]
        for t in TASKS
    ])
    im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=mat.min(), vmax=max(abs(mat).max(), 1e-6))
    ax.set_xticks(range(len(FEATURE_NAMES_CWRU)))
    ax.set_xticklabels(FEATURE_NAMES_CWRU, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(TASKS)))
    ax.set_yticklabels(TASKS, fontsize=9)
    ax.set_title(model, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "by_fault_type_per_task_heatmap.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_fault_type_per_task_heatmap.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/cwru_feature_importance/by_fault_type_per_task_heatmap.png


/tmp/ipykernel_95591/3755244491.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5 — EWC : permutation vs gradient saliency (normalisés)

In [5]:
ewc = data["EWC"]["global"]
perm = ewc["permutation_importance"]
grad = ewc["gradient_saliency"]

# Normalisation min-max pour comparaison sur même échelle
def minmax(d):
    vals = np.array(list(d.values()))
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.0 for k in d}
    return {k: (v - mn) / (mx - mn) for k, v in d.items()}

perm_n = minmax(perm)
grad_n = minmax(grad)

importances_ewc = {"Permutation (EWC)": perm_n, "Gradient Saliency (EWC)": grad_n}
fig = plot_feature_importance_comparison(
    importances_ewc,
    FEATURE_NAMES_CWRU,
    title="EWC — Permutation vs Gradient Saliency (normalisés) — CWRU by_fault_type",
)
plt.show()

print("Corrélation de Spearman entre les deux méthodes :")
from scipy.stats import spearmanr
rho, p = spearmanr(
    [perm_n[f] for f in FEATURE_NAMES_CWRU],
    [grad_n[f] for f in FEATURE_NAMES_CWRU],
)
print(f"  ρ = {rho:.3f}  (p = {p:.3f})")


Corrélation de Spearman entre les deux méthodes :
  ρ = 0.770  (p = 0.015)


/tmp/ipykernel_95591/786023600.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6 — HDC : permutation vs feature masking

In [6]:
hdc = data["HDC"]["global"]
perm_h = hdc["permutation_importance"]
mask_h = hdc["feature_masking_importance"]

perm_h_n = minmax(perm_h)
mask_h_n = minmax(mask_h)

importances_hdc = {"Permutation (HDC)": perm_h_n, "Masking (HDC)": mask_h_n}
fig = plot_feature_importance_comparison(
    importances_hdc,
    FEATURE_NAMES_CWRU,
    title="HDC — Permutation vs Feature Masking (normalisés) — CWRU by_fault_type",
)
plt.show()

from scipy.stats import spearmanr
rho, p = spearmanr(
    [perm_h_n[f] for f in FEATURE_NAMES_CWRU],
    [mask_h_n[f] for f in FEATURE_NAMES_CWRU],
)
print(f"Corrélation Spearman Permutation/Masking (HDC) : ρ = {rho:.3f}  (p = {p:.3f})")


Corrélation Spearman Permutation/Masking (HDC) : ρ = 0.496  (p = 0.175)


/tmp/ipykernel_95591/624161936.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 — Analyse : CWRU by_fault_type

### Questions clés

**Les 3 types de défaut ont-ils le même top-3 de features ?**

À compléter après exécution des cellules précédentes. Observer la heatmap (Section 4) :
- Si les colonnes `kurtosis`, `rms`, `crest` sont les plus colorées pour tous les modèles → invariance inter-tâche
- Si chaque type de défaut a un profil distinct → signe de domain shift fort

**`kurtosis` est-il stable ?**

- `kurtosis` est l'indicateur classique de choc mécanique (roulement défectueux)
- Sa présence dans le top-3 global (Section 2) confirme la validité physique du modèle
- Si `kurtosis` est dominant en EWC (gradient saliency) mais pas en KMeans → différence d'expressivité

### Notes pour le manuscrit

- `FIXME(gap1)` : confirmer que les features importantes sont cohérentes avec la littérature physique
- Features recommandées pour déploiement MCU : **kurtosis, rms, crest** (3 features sur 9)
- Réduction possible : passer de 9 à 3 features → réduction RAM ×3 (voir S11-23 ablation)
